In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd             # データ操作と分析
import numpy as np              # 数値計算
import re                       # 正規表現（テキスト前処理用）
import nltk                     # 自然言語処理ライブラリ
from nltk.corpus import stopwords # ストップワードリスト
from nltk.stem import WordNetLemmatizer # レマタイゼーション用

from sklearn.feature_extraction.text import TfidfVectorizer # TF-IDF特徴量変換
from sklearn.preprocessing import OneHotEncoder # One-Hot Encoding用
from scipy.sparse import hstack # 疎行列を水平方向に結合するため

from sklearn.model_selection import train_test_split, GridSearchCV # 訓練・検証データ分割、グリッドサーチ
from sklearn.metrics import f1_score # F1スコア計算
import xgboost as xgb           # XGBoostモデル

# NLTKのダウンロード（Kaggle環境で初回のみ必要、すでに実行済みならコメントアウト可）
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('omw-1.4')

In [ ]:
# データセットのパス（Kaggle Notebooks環境を想定）
train_path = '/kaggle/input/nlp-getting-started/train.csv'
test_path = '/kaggle/input/nlp-getting-started/test.csv'
sample_submission_path = '/kaggle/input/nlp-getting-started/sample_submission.csv'

# CSVファイルをDataFrameとして読み込む
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_submission_df = pd.read_csv(sample_submission_path)

# 読み込んだtrain_dfの最初の5行を表示して確認
train_df.head()

In [ ]:
# train_dfとtest_dfのkeyword列の欠損値を「no_keyword」で補完
train_df['keyword'].fillna('no_keyword', inplace=True)
test_df['keyword'].fillna('no_keyword', inplace=True)

# train_dfとtest_dfのlocation列の欠損値を「no_location」で補完
train_df['location'].fillna('no_location', inplace=True)
test_df['location'].fillna('no_location', inplace=True)

In [ ]:
# テキストクリーニング用の関数を定義
def clean_text(text):
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # URLを除去
    text = re.sub(r'<.*?>', '', text) # HTMLタグを除去
    emoji_pattern = re.compile(
        '['
        '\U0001F600-\U0001F64F' '\U0001F300-\U0001F5FF'
        '\U0001F680-\U0001F6FF' '\U0001F1E0-\U0001F1FF'
        '\U00002702-\U000027B0' '\U000024C2-\U0001F251'
        ']+',
        flags=re.UNICODE
    )
    text = emoji_pattern.sub(r'', text) # 絵文字を除去
    text = re.sub(r'[^\w\s]', '', text) # 句読点を除去
    text = re.sub(r'\d+', '', text) # 数字を除去
    text = text.lower() # 小文字に変換
    return text

# train_dfとtest_dfのtext列にクリーニング関数を適用
train_df['cleaned_text'] = train_df['text'].apply(clean_text)
test_df['cleaned_text'] = test_df['text'].apply(clean_text)


# ストップワード除去とレマタイゼーションを組み合わせた関数
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text_advanced(text):
    words = text.split()
    # ストップワード除去
    filtered_words = [word for word in words if word not in stop_words]
    # レマタイゼーション
    lemmatized_words = [lemmatizer.lemmatize(word) for word in filtered_words]
    return ' '.join(lemmatized_words)

# クリーニング済みテキストにストップワード除去とレマタイゼーションを適用
train_df['processed_text'] = train_df['cleaned_text'].apply(preprocess_text_advanced)
test_df['processed_text'] = test_df['cleaned_text'].apply(preprocess_text_advanced)

# 処理後のテキストの最初の5行を確認
train_df[['text', 'processed_text']].head()

In [ ]:
# --- TF-IDF特徴量の生成（ngram_rangeを(1,2)に設定） ---
# ngram_range=(1,2)で、ユニグラム（単一の単語）とバイグラム（2つの連続した単語）を特徴量として含める
tfidf_vectorizer = TfidfVectorizer(min_df=5, max_df=0.8, max_features=10000, ngram_range=(1, 2))

# train_dfとtest_dfのprocessed_text列を結合してTF-IDFを学習
# 訓練データとテストデータ全体で単語辞書を構築することで、一貫した特徴量が得られる
full_text = pd.concat([train_df['processed_text'], test_df['processed_text']])
tfidf_vectorizer.fit(full_text)

# 訓練データとテストデータにTF-IDF変換を適用
X_train_tfidf = tfidf_vectorizer.transform(train_df['processed_text'])
X_test_tfidf = tfidf_vectorizer.transform(test_df['processed_text'])


# --- keyword列のOne-Hot Encoding ---
# OneHotEncoderのインスタンスを作成（handle_unknown='ignore'でテストデータにないカテゴリを無視）
# sparse_output=Trueで疎行列を返すため、メモリ効率が良い
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)

# train_dfとtest_dfのkeyword列を結合してエンコーダーを学習
full_keywords = pd.concat([train_df[['keyword']], test_df[['keyword']]])
encoder.fit(full_keywords)

# 訓練データとテストデータにOne-Hot Encodingを適用
X_train_keyword = encoder.transform(train_df[['keyword']])
X_test_keyword = encoder.transform(test_df[['keyword']])


# --- 全ての特徴量を結合 ---
# scipy.sparse.hstackを使って疎行列を横に結合し、最終的な特徴量セットを作成
X_train_combined = hstack([X_train_tfidf, X_train_keyword])
X_test_combined = hstack([X_test_tfidf, X_test_keyword])

# ターゲット変数（ラベル）をy_trainに格納
y_train = train_df['target']

# 結合後の特徴量行列の形状を確認したい場合は以下のコメントを外す
# X_train_combined.shape
# X_test_combined.shape

In [ ]:
# XGBoost分類器のインスタンスを作成
# 'predictor' パラメータを削除し、警告を回避。
# 'tree_method' に 'hist' を指定することで、GPUが利用可能な環境では自動的にGPUが使われます。
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False, # 新しいバージョンで推奨される設定
    random_state=42, # 結果の再現性を確保
    tree_method='hist' # GPUが利用可能なら高速化
)

# チューニングするハイパーパラメータの範囲を定義
param_grid = {
    'n_estimators': [200, 300], # 決定木の数
    'learning_rate': [0.05, 0.1], # 学習率
    'max_depth': [5, 7] # 決定木の最大深さ
}

# GridSearchCVのインスタンスを作成
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=3, # 3分割交差検定
    scoring='f1', # F1スコアを最適化
    n_jobs=-1, # 全てのCPUコアを利用
    verbose=2 # 実行中の詳細を表示
)

# グリッドサーチを実行
grid_search.fit(X_train_combined, y_train)

# 最適なハイパーパラメータと、そのときのベストスコアを確認したい場合は以下のコメントを外す
# grid_search.best_params_
# grid_search.best_score_

# 最適なモデルを取得
best_xgb_model = grid_search.best_estimator_

In [ ]:
# 最適なXGBoostモデルでテストデータに対して予測を実行
final_predictions_xgb = best_xgb_model.predict(X_test_combined)

# 提出用DataFrameを作成
# sample_submission_dfをコピーし、'target'列を予測結果で更新
submission_df_xgb = sample_submission_df.copy()
submission_df_xgb['target'] = final_predictions_xgb

# 提出ファイルの最初の5行を確認
submission_df_xgb.head()

In [ ]:
# 提出ファイルをCSV形式で保存
# index=Falseを指定して、DataFrameのインデックスがCSVに含まれないようにする
# ファイル名は'submission.csv'とし、Kaggle提出時に自動で認識されるようにする
submission_df_xgb.to_csv('submission.csv', index=False)